# 97 — Campaign B post-M2 pipeline（95 相当の消化レーン）

**89 と並走可（95 と同じ思想）。待ち行列が増えてもよい（backlog OK）。**

既に溜まっている M2-ready / READY_FOR_M3 を消化する consumer。

- デフォルト: `drain_existing_backlog=True` / `skip_screening=True`
  → `advance → M3 → pre_m6 → obligations → M6`（`run_pipeline_to_m6`）
- 対象: SELECTED で `ADVANCE.json` が `READY_FOR_M3`、または `m2_binding` が `READY` / `READY_SHARED`
- `M2_READY.json` は検出のみ（待機ゲートではない）
- 高 drain knobs: `MAX_ROUNDS=20+`, `MAX_M3_SESSIONS` / `MAX_PRE_M6_PACKAGES` /
  `MAX_OBLIGATION_PACKAGES` / `MAX_M6_PACKAGES` ≈ 16, `MAX_QUEUE=2000`
- screening を回したいときだけ `DRAIN_EXISTING_BACKLOG=False`（end_to_end 経路）
- GPU lane lease: `campaign_b/_locks/gpu_lane.json` — **96 と同時フル起動しない**
  （第二側は `GpuLaneHeldError`）
- **M3 auto-strip（既定 ON）:** セッション開始時に COMPLETE+downstream をフル安全
  スキャンして strip。各ラウンド後も増分 strip。reports は残す。
- **Keep-latest（既定 ON）:** セッション開始時に **全** `runs/M3-*` へフル keep-latest
  （このバッチで触れない古い M3 の ckpt 積み上げも回収）。加えて各 M3 セッションの
  resume 前・後でも per-run trim（ckpt_000001…N 再蓄積防止）。
- knobs: `AUTO_STRIP_M3_CHECKPOINTS=True`, `AUTO_KEEP_LATEST_M3_CHECKPOINT=True`,
  `PERSIST_M3_CAP_GIB=32.0`。詳細: `docs/campaign_b_m3_storage_reclaim.md`
- **停止後の再実行:** セル 3 を再実行すれば backlog を再開。`stop_reason` を確認:
  - `DRAINED_OR_IDLE` — 実行可能キューが空（正常 idle）
  - `STUCK_BACKLOG` — 試行したが進捗なし（連続 idle 上限）→ 再実行 / エラー確認
  - `NO_ATTEMPTS_WITH_BACKLOG` — backlog あるのに試行 0（CUDA/設定）
  - `MAX_ROUNDS` — ラウンド上限。再実行で続きを消化

設計: `docs/campaign_b_parallel_split_design.md`


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'post_m2_pipeline.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/post_m2_pipeline.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ['VALIDATED_RG_DISABLE_SESSION_WALLCLOCK'] = '1'
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M3_ALLOW_CODE_DRIFT', '1')
os.environ.setdefault('VALIDATED_RG_M4_ALLOW_CODE_DRIFT', '1')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)
if not torch.cuda.is_available():
    raise RuntimeError('Notebook 97 requires CUDA (single GPU lane).')
print('GPU', torch.cuda.get_device_properties(0).name)


In [ ]:
from src.runtime import validate_persistent_root
from src.campaign_b.post_m2_pipeline import find_m2_ready_markers

validate_persistent_root()
print('M2_READY markers (informational):', json.dumps(
    find_m2_ready_markers(PERSIST_ROOT), indent=2, default=str,
))

# Defaults = 95-equivalent drain of existing M2-ready / READY_FOR_M3 backlog.
DRAIN_EXISTING_BACKLOG = True   # False → Phase-1 end_to_end (optional screening)
SKIP_SCREENING = True           # keep True for consumer-only; ignored when draining
MAX_ROUNDS = 20                 # re-scan while backlog remains
MAX_ADVANCE = None              # None = all discovered SELECTED
MAX_M3_SESSIONS = 16
MAX_PRE_M6_PACKAGES = 16
MAX_STAGE_SESSIONS = 6
MAX_OBLIGATION_PACKAGES = 16
MAX_M6_PACKAGES = 16
MAX_QUEUE = 2000
ONLY_CAMPAIGN = None            # or pin one M7-... campaign id
# After M4+ consumes M3: strip M3 checkpoints (fail-closed). Default ON.
# Session start always runs a full safe strip scan (backlog reclaim).
# See docs/campaign_b_m3_storage_reclaim.md
AUTO_STRIP_M3_CHECKPOINTS = True
# Session start: full keep-latest over ALL runs/M3-* (not only this batch).
# Also per M3 session before/after run_one_gpu_m3 (stops mid-run re-accumulation).
AUTO_KEEP_LATEST_M3_CHECKPOINT = True
# Cap total runs/M3-* size (GiB). None disables. Paperspace 15GB included + billing.
PERSIST_M3_CAP_GIB = 32.0
# Used only when DRAIN_EXISTING_BACKLOG=False:
SELECTED_BACKLOG_TARGET = 8
SCREENING_CHUNK_SIZE = 32

print(json.dumps({
    'DRAIN_EXISTING_BACKLOG': DRAIN_EXISTING_BACKLOG,
    'SKIP_SCREENING': SKIP_SCREENING,
    'MAX_ROUNDS': MAX_ROUNDS,
    'MAX_ADVANCE': MAX_ADVANCE,
    'MAX_M3_SESSIONS': MAX_M3_SESSIONS,
    'MAX_PRE_M6_PACKAGES': MAX_PRE_M6_PACKAGES,
    'MAX_STAGE_SESSIONS': MAX_STAGE_SESSIONS,
    'MAX_OBLIGATION_PACKAGES': MAX_OBLIGATION_PACKAGES,
    'MAX_M6_PACKAGES': MAX_M6_PACKAGES,
    'MAX_QUEUE': MAX_QUEUE,
    'ONLY_CAMPAIGN': ONLY_CAMPAIGN,
    'AUTO_STRIP_M3_CHECKPOINTS': AUTO_STRIP_M3_CHECKPOINTS,
    'AUTO_KEEP_LATEST_M3_CHECKPOINT': AUTO_KEEP_LATEST_M3_CHECKPOINT,
    'PERSIST_M3_CAP_GIB': PERSIST_M3_CAP_GIB,
}, indent=2))


In [ ]:
from src.campaign_b.post_m2_pipeline import run_post_m2_pipeline

SESSION = run_post_m2_pipeline(
    persistent_root=PERSIST_ROOT,
    project_root=PROJECT_ROOT,
    drain_existing_backlog=DRAIN_EXISTING_BACKLOG,
    skip_screening=SKIP_SCREENING,
    selected_backlog_target=SELECTED_BACKLOG_TARGET,
    screening_chunk_size=SCREENING_CHUNK_SIZE,
    max_rounds=MAX_ROUNDS,
    max_advance=MAX_ADVANCE,
    max_m3_sessions=MAX_M3_SESSIONS,
    max_pre_m6_packages=MAX_PRE_M6_PACKAGES,
    max_stage_sessions=MAX_STAGE_SESSIONS,
    max_obligation_packages=MAX_OBLIGATION_PACKAGES,
    max_m6_packages=MAX_M6_PACKAGES,
    max_queue=MAX_QUEUE,
    only_campaign_run_id=ONLY_CAMPAIGN,
    auto_strip_m3_checkpoints=AUTO_STRIP_M3_CHECKPOINTS,
    auto_keep_latest_m3_checkpoint=AUTO_KEEP_LATEST_M3_CHECKPOINT,
    persist_m3_cap_gib=PERSIST_M3_CAP_GIB,
)
print(json.dumps({
    'session_id': SESSION.get('session_id'),
    'mode': SESSION.get('mode'),
    'drain_existing_backlog': SESSION.get('drain_existing_backlog'),
    'skip_screening': SESSION.get('skip_screening'),
    'auto_strip_m3_checkpoints': SESSION.get('auto_strip_m3_checkpoints'),
    'auto_keep_latest_m3_checkpoint': SESSION.get('auto_keep_latest_m3_checkpoint'),
    'persist_m3_cap_gib': SESSION.get('persist_m3_cap_gib'),
    'stop_reason': SESSION.get('stop_reason'),
    'remaining_runnable': SESSION.get('remaining_runnable'),
    'stuck_diagnostics': SESSION.get('stuck_diagnostics'),
    'm3_reclaim': SESSION.get('m3_reclaim'),
    'm2_ready_count': SESSION.get('m2_ready_count'),
    'pipeline_to_m6': SESSION.get('pipeline_to_m6'),
    'end_to_end': SESSION.get('end_to_end'),
    'ledger': str(PERSIST_ROOT / 'campaign_b' / '_post_m2' / 'LATEST_POST_M2_SESSION.json'),
    'certification_status': SESSION.get('certification_status'),
    'claim_scope': SESSION.get('claim_scope'),
    'note': SESSION.get('note'),
}, indent=2, ensure_ascii=False, default=str))
# Re-run this cell after stop_reason in {STUCK_BACKLOG, MAX_ROUNDS,
# NO_ATTEMPTS_WITH_BACKLOG} once CUDA/disk/errors are fixed. Backlog OK.
# On STUCK_BACKLOG, stuck_diagnostics.m3_errors shows why attempts failed.
